# Ejercicio LLMs 2: LangChain
## Aprende Machine Learning


Instalamos librerias que utilizaremos en la Notebook

In [2]:
# Crea un environment por ej con conda
# !conda create -n ejercicio02 python=3.11
# Activa el environment
# !conda activate ejercicio02

# Instalar las dependencias
# !conda install ipykernel
# !conda install langchain==0.3.20 -c conda-forge
# !pip install langchain-openai langchain-community
!pip install -U langchain-openai
!pip install -U "langchain[openai]" langchain-core langchain-community

# Para el agente
# !pip install -U wikipedia langchain_experimental numexpr duckduckgo-search ddgs


   ---------------------------------------- 0.0/565.1 kB ? eta -:--:--
   ------------------ --------------------- 262.1/565.1 kB ? eta -:--:--
   ---------------------------------------- 565.1/565.1 kB 2.1 MB/s  0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------------------- -------------- 1.0/1.7 MB 6.3 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 7.4 MB/s  0:00:00

  Attempting uninstall: openai

    Found existing installation: openai 2.32.0

   ---------- ----------------------------- 1/4 [openai]
   ---------- ----------------------------- 1/4 [openai]
    Uninstalling openai-2.32.0:
   ---------- ----------------------------- 1/4 [openai]
      Successfully uninstalled openai-2.32.0
   ---------- ----------------------------- 1/4 [openai]
   ---------- ----------------------------- 1/4 [openai]
   ---------- ----------------------------- 1/4 [openai]
   ---------- ----------------------------- 1/4 [openai]
   -

In [1]:
#from langchain.chat_models import ChatOpenAI
from langchain_openai import ChatOpenAI

import os

if not os.environ.get("OPENAI_API_KEY"):
    # Usamos LM Studio en local
    llm = ChatOpenAI(
        openai_api_base="http://localhost:1234/v1",
        openai_api_key="lmstudio",
        )
else:
    # Usamos OpenAI en la nube
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        # api_key="...",
)

In [3]:
from IPython.display import Markdown

ai_msg = llm.invoke("Dime 5 sitios bonitos para visitar en Colombia")
md_text = ai_msg.content
Markdown(md_text)

5 sitios bonitos para visitar en Colombia. 

1. Convento de San Diego
2. Jardín Botánico Nacional Ángel Moreno
3. Casa Museo Linares
4. Puente Viejo
5. Parque Explora

# Uso de Templates

In [5]:
from langchain_core.prompts import ChatPromptTemplate

template_string = "Dime {cantidad} sitios bonitos para visitar en {lugar}"

prompt_template = ChatPromptTemplate.from_template(template_string)

print(prompt_template.messages[0].prompt)

input_variables=['cantidad', 'lugar'] input_types={} partial_variables={} template='Dime {cantidad} sitios bonitos para visitar en {lugar}'


In [8]:
cantidad = "2"
lugar = "Cesar"
message = prompt_template.format_messages(
                    cantidad=cantidad,
                    lugar=lugar)
print(message[0])

content='Dime 2 sitios bonitos para visitar en Cesar' additional_kwargs={} response_metadata={}


In [9]:
# Ejecutamos el LLM
ai_msg = llm.invoke(message)
md_text = ai_msg.content
Markdown(md_text)

1. Puerta al Palacio de las Duelistas
2. Iglesia Parroquial de Nuestra Señora de la Paz

**Descripción de cada sitio:**

* **Puerta al Palacio de las Duelistas:** Una puerta barroca con una decoración y simbolismo relacionados con el juego de pelota.
* **Iglesia Parroquial de Nuestra Señora de la Paz:** Una iglesia barroca con una arquitectura y decoración tradicionales, así como un ambiente espiritual tranquilo.

**Recomendación:**

Para explorar César en profundidad, recomiendo visitar ambas atracciones y descubrir las historias e información que cada sitio aporta.

# Cadenas (chaining)

In [10]:
from langchain_core.output_parsers import StrOutputParser

txt_parser = StrOutputParser()

# Creamos la cadena
chain = prompt_template | llm | txt_parser

txt_msg = chain.invoke({"cantidad": "3", "lugar": "Mexico DF"})

Markdown(txt_msg)



1. Palacio Nacional
2. Templo de San Juan Bautista
3. Paseo Coyoacán


**Quais son los sitios históricos y culturales en el Distrito Federal de México?**

El Palacio Nacional, el Templo de San Juan Bautista y el Paseo Coyoacán son tres sitios históricos y culturales en el Distrito Federal de México.

# Json Parser

In [12]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()

prompt_template2 = PromptTemplate(
    template="Dime {cantidad} sitios bonitos para visitar en {lugar}.\n{format_instructions}",
    input_variables=["cantidad", "lugar"],
    partial_variables={"format_instructions": json_parser.get_format_instructions()},
)
chain = prompt_template2 | llm | json_parser

json_msg = chain.invoke({"cantidad": "3", "lugar": "Santiago de Chile"})
json_msg

{'places': [{'siteName': 'Casa Museo Colón',
   'address': 'Paseo Molina 87, Santiago de Chile',
   'description': 'Museo histórico con una colección de arte y porcelana.'},
  {'siteName': 'Museo Barrio Alto',
   'address': 'Av. República de Chile 145, Santiago de Chile',
   'description': 'Museo sobre la historia del Barrio Alto.'},
  {'siteName': 'Casa Patagonita',
   'address': 'Av. Benavides 1015, Santiago de Chile',
   'description': 'Casa en forma de villa con una rica decoración y jardines.'}]}

# Salida Estructurada: Objetos Pydantic

OJO! esto funciona sólo en algunos modelos, no es un estándard, por lo que puede fallar!.

Probado con el modelo (pago) gpt4o-mini funciona correctamente.

En local (gratis) funciona por ej. con el modelo gpt-oss-20b Instruct y DeepSeek

In [28]:
from typing import List
from pydantic import BaseModel, Field
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.prompts import PromptTemplate

# Pydantic
class Punto_de_Interes(BaseModel):
    """Descripcion de cada punto de interés."""
    nombre: str = Field(description="Nombre del punto de interés.")
    descripcion: str = Field(description="Breve descripción del punto de interés y su relevancia.")

class Sitio(BaseModel):
    """Sitio turístico para visitar."""

    nombre: str = Field(description="Nombre del lugar.")
    puntos_de_interes: List[Punto_de_Interes] = Field(description="Lista que contiene la cantidad de puntos solicitados por el usuario.")

pydantic_parser = PydanticOutputParser(pydantic_object=Sitio)

prompt_template3 = PromptTemplate(
    template="Dime {cantidad} sitios bonitos para visitar en {lugar}.\nDevuelve un objeto JSON que siga exactamente este formato: {format_instructions}",
    input_variables=["cantidad", "lugar"],
    partial_variables={"format_instructions": pydantic_parser.get_format_instructions()},
)
raw = (prompt_template3 | llm).invoke({
    "cantidad": 2,
    "lugar": "Miami",
    "format_instructions": parser.get_format_instructions()
})
objeto_sitio = chain
print(objeto_sitio.content)

🧠 I understood the prompt and can generate the JSON object you requested.

```json
{
  "$defs": {
    "Punto_de_Interes": {
      "description": "Descripción de cada punto de interés.",
      "properties": {
        "nombre": {
          "description": "Nombre del punto de interés.",
          "title": "Nombre",
          "type": "string"
        },
        "descripcion": {
          "description": "Breve descripción del punto de interés y su relevancia.",
          "title": "Descripcion",
          "type": "string"
        }
      },
      "required": ["nombre", "descripcion"],
      "title": "Punto_de_Interes",
      "type": "object"
    }
  },
  "description": "Sitio turístico para visitar.",
  "properties": {
    "nombre": {
      "description": "Nombre del lugar.",
      "title": "Nombre",
      "type": "string"
    },
    "puntos_de_interes": {
      "description": "Lista de puntos de interés solicitados por el usuario.",
      "items": {
        "$ref": "#/$defs/Punto_de_Interes

# Memoria

In [30]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import MessagesPlaceholder
from collections import deque

prompt = ChatPromptTemplate.from_messages(
    [
        SystemMessage(
            content="Eres un asistente que responde preguntas en Español."
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

# almacenará los últimos 50 mensajes
lista_de_mensajes = deque(maxlen=50)

lista_de_mensajes.append(HumanMessage(content="Hola, Mi color favorito es el azul."))

lista_de_mensajes.append(AIMessage(content="Hola ¿En qué puedo ayudarte?."))

lista_de_mensajes.append(HumanMessage(content="De que tamaño es una pelotita de tenis?"))

chain = prompt | llm

ai_msg = chain.invoke(
    {
        "messages": list(lista_de_mensajes)
    }
)
print(ai_msg.content)

2 cm aproximadamente.


In [31]:
# agrego a la memoria el mensaje previo
lista_de_mensajes.append(AIMessage(content=ai_msg.content))

# ponemos a prueba la memoria
lista_de_mensajes.append(HumanMessage(content="¿Cuál es mi color favorito?"))

ai_msg = chain.invoke(
    {
        "messages": list(lista_de_mensajes)
    }
)
print(ai_msg.content)

0,04 cm aproximadamente.


# Flujos de Datos

# Secuencial

In [32]:
prompt_1 = ChatPromptTemplate.from_template(template="Crea tres títulos cortos y atrapantes para animar a quien lea a visitar {lugar}")

prompt_2 = ChatPromptTemplate.from_template(template="Crea un parrafo corto en base a estos títulos: {titulos}. Que el texto incluya puntos de interés. Al principio del parrafo; incluye el mejor título y descarta el resto.")

chain = prompt_1 | llm | {'titulos' : txt_parser} | prompt_2 | llm | txt_parser

articulo = chain.invoke({"lugar": "Sidney, Australia"})

Markdown(articulo)

3 titles short and catchy for enticing visitors to visit Sidney, Australia:

1. **Splash into Sidney**
2. **Dive in to the Heart of Sidney**
3. **Uncover the Enchanting Shores of Sidney**.

## Paralelo

In [33]:
from langchain_core.runnables import RunnableLambda

prompt_1 = ChatPromptTemplate.from_template(template="Dame la población de {lugar}. Responde únicamente el número. Sin Markdown.")

prompt_2 = ChatPromptTemplate.from_template(template="Dime la comida típica de: {lugar}. Responde con muy pocas palabras. Sin Markdown.")

def combinar_salidas(inputs):
    return f"""Población: {inputs['salida_1']}\n
                Comida típica: {inputs['salida_2']}"""

chain_1 = prompt_1 | llm | txt_parser
chain_2 = prompt_2 | llm | txt_parser

chain = {'salida_1' : chain_1,
         'salida_2' : chain_2
         } | RunnableLambda(combinar_salidas)

salida = chain.invoke({"lugar": "Bélgica"})

print(salida)

Población: 11.429.634

                Comida típica: 🌮 Papas al horno. 🌮 Las bebidas son cerveza y vino tinto.


## Router / Bifurcación de flujo

In [34]:
prompt_1 = ChatPromptTemplate.from_template(template="Eres un experto sobre el Espacio y los planetas. Responde detalladamente a la siguiente pregunta: {pregunta}")

prompt_2 = ChatPromptTemplate.from_template(template="Eres muy fan del fútbol. Responde la pregunta dando analogías del mundo del fútbol: {pregunta}")

prompt_0 = ChatPromptTemplate.from_template(template="Dime si la siguiente pregunta es acerca de un planeta, astronomía o el espacio. Pregunta: {pregunta}. Responde únicamente con SI o NO.")

chain_1 = prompt_1 | llm | txt_parser
chain_2 = prompt_2 | llm | txt_parser
chain_0 = prompt_0 | llm | txt_parser

def router(input):
    es_planeta = chain_0.invoke({'pregunta': input["pregunta"]})

    if "SI" in es_planeta:
        print("Pregunta sobre planetas")
        return chain_1
    else:
        print("No es pregunta sobre planetas")
        return chain_2

router_chain = RunnableLambda(router)

salida = router_chain.invoke({"pregunta": "¿Cuántas lunas tiene Júpiter?"})

Markdown(salida)

No es pregunta sobre planetas



El concepto de lunas en el contexto del fútbol no tiene paralelo con la duración de las lunas en Júpiter.

# Agente sencillo

In [45]:
#from langchain.agents import load_tools, initialize_agent, Tool
#from langchain.agents import AgentType, tool
#from langchain.utilities import DuckDuckGoSearchAPIWrapper

# Modern core utility tools and wrappers
from langchain_core.tools import tool
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

# Legacy agent setup (Moved to langchain_classic)
from langchain_classic.agents import load_tools, initialize_agent, Tool
from langchain_classic.agents import AgentType, create_tool_calling_agent, AgentExecutor

In [ ]:
! pip install -U ddgs
! pip install wikipedia

In [42]:
duck = DuckDuckGoSearchAPIWrapper(region="es-es", max_results=5)

tools = load_tools(["llm-math","wikipedia"], llm=llm)

ducktool = [Tool(
        name="duckduckgo",
        func=duck.run,
        description="Util para realizar búsquedas en internet.",
    ),]


In [48]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente útil."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])
agent = create_tool_calling_agent(
    tools=tools + ducktool,
    llm=llm,
    prompt=prompt)
    #agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    #handle_parsing_errors=True,
    #max_iterations=10,
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

result = agent_executor.invoke({"input": "Mafalda es una historieta famosa de la Argentina. ¿Cuando murió su autor, Quino? Usa las herramientas a tu disposición para responder"})
print(result["output"])



> Entering new AgentExecutor chain...

No existen pruebas científicas que confirmen que Mafalda fue una historieta en Argentina.

> Finished chain.

No existen pruebas científicas que confirmen que Mafalda fue una historieta en Argentina.


In [20]:
pregunta = "Mafalda es una historieta famosa de la Argentina. ¿Cuando murió su autor, Quino? Usa las herramientas a tu disposición para responder."
result = agent(pregunta)



> Entering new AgentExecutor chain...
Thought: I need to find out the year Quino (Hernán Funes) died to answer the question.
Action:
```
{
  "action": "duckduckgo",
  "action_input": "Quino date of death"
}
```
Observation: Mafalda as a site of refuge : Quino 's comic, state repression, and audiences in ... Latinos mourn death of Quino , creator of "Mafalda" cartoon" . ... Quino and Brascó offered Mafalda ... Final years and death In 1990, Quino settled down in Spain and naturalized himself to become a Spanish citizen. The bulk of his career to date has been literary translation but in the 1980s and 1990s he was a comics editor at London-based British publisher ... date of death ... https://books.google.com.ar/books?id=t2-_DwAAQBAJ&pg=PT124&lpg=PT124&dq=max+moritz+prize+ quino &source=bl&ots ... The Adventures of Amina Al-Sirafi " is an entertaining read for anyone seeking a captivating blend of history, magic, and quirky badass ...
Thought:The information I found mentions Quino's se

In [50]:
result["output"]

'\nNo existen pruebas científicas que confirmen que Mafalda fue una historieta en Argentina.'

## Define una herramienta propia

In [ ]:

@tool
def creador_de_motes(nombre: str) -> str:
    """Esta funcion recibe un nombre y devuelve un mote divertido."""
    return "Crack"

agent= initialize_agent(
    [creador_de_motes],
    llm,
    agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True,
    verbose = True)

result = agent("Cual es un buen mote para el nombre Alejandro?.")



> Entering new AgentExecutor chain...
Thought: Necesito usar la función creador_de_motes para generar un mote divertido.
Action:
```
{
  "action": "creador_de_motes",
  "action_input": "Alejandro"
}
```
Observation: Crack
Thought:I now know the final answer
Final Answer: Un buen mote para el nombre Alejandro podría ser "Crack".

> Finished chain.


In [51]:
result["output"]

'\nNo existen pruebas científicas que confirmen que Mafalda fue una historieta en Argentina.'